<a href="https://colab.research.google.com/github/naveen-2501/Project_Manager/blob/machine_learning(FCC)/fcc_book_recommendation_knn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import libraries (you may add additional imports but you may not have to)
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2025-07-19 11:37:28--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 172.67.70.149, 104.26.3.33, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: ‘book-crossings.zip’

book-crossings.zip  100%[===================>]  24.88M  21.7MB/s    in 1.1s    

2025-07-19 11:37:30 (21.7 MB/s) - ‘book-crossings.zip’ saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

In [4]:
# add your code here - consider creating a new cell for each section of code
ratings_filtered = ratings.groupby('User-ID').filter(lambda x: len(x) >= 200)
ratings_filtered = ratings_filtered.groupby('ISBN').filter(lambda x: len(x) >= 100)

# Merge ratings with books to get book titles
ratings_with_titles = pd.merge(ratings_filtered, books, on='ISBN')

# Create user-book matrix (pivot table)
user_book_matrix = ratings_with_titles.pivot_table(index='Book-Title', columns='User-ID', values='Book-Rating').fillna(0)

# Train Nearest Neighbors model
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(user_book_matrix.values)

# Create a mapping of book title to index
book_titles = user_book_matrix.index.tolist()

NameError: name 'ratings' is not defined

In [6]:
# function to return recommended books - this will be tested
def get_recommends(book = ""):

  try:
        # Get the index of the book
        book_idx = book_titles.index(book_title)
    except ValueError:
        return f"'{book_title}' not found in dataset."

    # Get distances and indices of 6 nearest neighbors (including itself)
    distances, indices = model.kneighbors([user_book_matrix.iloc[book_idx].values], n_neighbors=6)

    # Collect recommended books (excluding the first one since it's the book itself)
    recommended = []
    for i in range(1, len(indices[0])):
        recommended.append([book_titles[indices[0][i]], distances[0][i]])

    return [book_title, recommended]

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 7)

In [ ]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2):
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()